# Практическая работа №2: Группировка, обработка пропусков и объединение данных

Цель работы: Освоить продвинутые операции с данными — группировку и агрегацию, обработку пропущенных значений, слияние и переименование таблиц.

In [2]:
import pandas as pd
import numpy as np

path = '../shared/titanic.csv'
df = pd.read_csv(path)
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


## 1. Группировка и сортировка данных

### 1.1 Группировка: `.groupby()`

Группировка позволяет разделить данные на группы по значению одного или нескольких столбцов, а затем применить к ним агрегирующие функции (сумма, среднее, количество и т.д.).

Концепция Split-Apply-Combine: Split — разделить данные на группы, Apply — применить функцию к каждой группе, Combine — объединить результаты.

In [3]:
# Средняя стоимость билета по классам
print(df.groupby('Pclass')['Fare'].mean())

Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64


Группировка с несколькими агрегатами

In [4]:
# Минимальная, максимальная и средняя стоимость билета по классам
print(df.groupby('Pclass')['Fare'].agg(['min', 'max', 'mean', 'count']))

        min       max       mean  count
Pclass                                 
1       0.0  512.3292  84.154687    216
2       0.0   73.5000  20.662183    184
3       0.0   69.5500  13.675550    491


Группировка по нескольким столбцам

In [5]:
# Средняя стоимость билета по классу и полу
print(df.groupby(['Pclass', 'Sex'])['Fare'].mean())

Pclass  Sex   
1       female    106.125798
        male       67.226127
2       female     21.970121
        male       19.741782
3       female     16.118810
        male       12.661633
Name: Fare, dtype: float64


Применение собственной функции

In [6]:
def fare_range(series):
    return series.max() - series.min()

print(df.groupby('Pclass')['Fare'].apply(fare_range))

Pclass
1    512.3292
2     73.5000
3     69.5500
Name: Fare, dtype: float64


### 1.2 Сортировка: `.sort_values()` и `.sort_index()`

Сортировка по значениям столбцов

In [7]:
# Сортировка по возрасту (по возрастанию)
print(df.sort_values('Age').head())

# Сортировка по стоимости билета (по убыванию)
print(df.sort_values('Fare', ascending=False).head())

# Сортировка по нескольким столбцам
print(df.sort_values(['Pclass', 'Fare'], ascending=[True, False]).head())

     PassengerId  Survived  Pclass                             Name     Sex  \
803          804         1       3  Thomas, Master. Assad Alexander    male   
755          756         1       2        Hamalainen, Master. Viljo    male   
644          645         1       3           Baclini, Miss. Eugenie  female   
469          470         1       3    Baclini, Miss. Helene Barbara  female   
78            79         1       2    Caldwell, Master. Alden Gates    male   

      Age  SibSp  Parch  Ticket     Fare Cabin Embarked  
803  0.42      0      1    2625   8.5167   NaN        C  
755  0.67      1      1  250649  14.5000   NaN        S  
644  0.75      2      1    2666  19.2583   NaN        C  
469  0.75      2      1    2666  19.2583   NaN        C  
78   0.83      0      2  248738  29.0000   NaN        S  
     PassengerId  Survived  Pclass                                Name  \
258          259         1       1                    Ward, Miss. Anna   
737          738         1   

Сортировка после группировки

In [8]:
# Группировка и сортировка результата
result = df.groupby('Pclass')['Fare'].mean().sort_values(ascending=False)
print(result)

Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64


Сортировка внутри групп

In [9]:
# Отсортировать данные внутри каждого класса по стоимости билета
df_sorted = df.sort_values(['Pclass', 'Fare'], ascending=[True, False])
print(df_sorted.groupby('Pclass').head(2))  # первые 2 записи из каждой группы

     PassengerId  Survived  Pclass                                Name  \
258          259         1       1                    Ward, Miss. Anna   
679          680         1       1  Cardeza, Mr. Thomas Drake Martinez   
72            73         0       2                Hood, Mr. Ambrose Jr   
120          121         0       2         Hickman, Mr. Stanley George   
159          160         0       3          Sage, Master. Thomas Henry   
180          181         0       3        Sage, Miss. Constance Gladys   

        Sex   Age  SibSp  Parch        Ticket      Fare        Cabin Embarked  
258  female  35.0      0      0      PC 17755  512.3292          NaN        C  
679    male  36.0      0      1      PC 17755  512.3292  B51 B53 B55        C  
72     male  21.0      0      0  S.O.C. 14879   73.5000          NaN        S  
120    male  21.0      2      0  S.O.C. 14879   73.5000          NaN        S  
159    male   NaN      8      2      CA. 2343   69.5500          NaN        S  
1

## 2. Типы данных и обработка пропущенных значений

### 2.1 Типы данных

Pandas автоматически определяет типы данных при чтении, но иногда требуется явное преобразование.

In [10]:
print(df.dtypes)

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


Преобразование типов: `.astype()`

In [11]:
# Преобразовать столбец в строковый тип
df['PassengerId_str'] = df['PassengerId'].astype(str)

# Преобразовать в целочисленный (сначала заполнив пропуски)
df['Age_int'] = df['Age'].fillna(0).astype(int)

# Преобразовать в категориальный (для экономии памяти)
df['Embarked'] = df['Embarked'].astype('category')

print(df.dtypes)

PassengerId           int64
Survived              int64
Pclass                int64
Name                    str
Sex                     str
Age                 float64
SibSp                 int64
Parch                 int64
Ticket                  str
Fare                float64
Cabin                   str
Embarked           category
PassengerId_str         str
Age_int               int64
dtype: object


### 2.2 Обработка пропущенных значений (NaN)

В реальных данных часто встречаются пропуски (missing values), обозначаемые как `NaN` (Not a Number).

In [12]:
# Количество пропусков в каждом столбце
print(df.isnull().sum())

# Есть ли хотя бы один пропуск?
print(df.isnull().any())

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                177
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin              687
Embarked             2
PassengerId_str      0
Age_int              0
dtype: int64
PassengerId        False
Survived           False
Pclass             False
Name               False
Sex                False
Age                 True
SibSp              False
Parch              False
Ticket             False
Fare               False
Cabin               True
Embarked            True
PassengerId_str    False
Age_int            False
dtype: bool


Заполнение пропусков: `.fillna()`

In [13]:
# Заполнить конкретный столбец средним значением
df_age_filled = df.copy()
df_age_filled['Age'] = df_age_filled['Age'].fillna(df_age_filled['Age'].mean())

# Заполнить предыдущим значением (forward fill)
df_embarked_filled = df.copy()
df_embarked_filled['Embarked'] = df_embarked_filled['Embarked'].ffill()

# Заполнить разными значениями для разных столбцов
df_filled_dict = df.copy()
df_filled_dict['Age'] = df_filled_dict['Age'].fillna(df_filled_dict['Age'].median())
df_filled_dict['Embarked'] = df_filled_dict['Embarked'].astype('object').fillna('Unknown')

print(df_filled_dict[['Age', 'Embarked']].isnull().sum())

Age         0
Embarked    0
dtype: int64


Удаление пропусков: `.dropna()`

In [14]:
# Удалить строки с любыми NaN
df_dropped = df.dropna()
print('Строк без NaN:', df_dropped.shape[0])

# Удалить строки, где NaN в конкретных столбцах
df_dropped_subset = df.dropna(subset=['Age', 'Embarked'])
print('Строк без NaN в Age/Embarked:', df_dropped_subset.shape[0])

# Удалить столбцы с NaN
df_dropped_cols = df.dropna(axis=1)
print('Столбцы без NaN:', list(df_dropped_cols.columns))

Строк без NaN: 183
Строк без NaN в Age/Embarked: 712
Столбцы без NaN: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'SibSp', 'Parch', 'Ticket', 'Fare', 'PassengerId_str', 'Age_int']


Замена значений: `.replace()`

In [15]:
# Заменить несколько значений (сначала переводим категориальный столбец обратно в строковый)
df_replaced = df.astype({'Embarked': 'object'}).replace({'Embarked': {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}})
print(df_replaced['Embarked'].value_counts(dropna=False))

# Заменить NaN на значение
df_replaced_nan = df_replaced.replace(np.nan, 0)
print(df_replaced_nan.isnull().sum())

Embarked
Southampton    644
Cherbourg      168
Queenstown      77
NaN              2
Name: count, dtype: int64
PassengerId        0
Survived           0
Pclass             0
Name               0
Sex                0
Age                0
SibSp              0
Parch              0
Ticket             0
Fare               0
Cabin              0
Embarked           0
PassengerId_str    0
Age_int            0
dtype: int64


## 3. Переименование и объединение данных

### 3.1 Переименование столбцов и индексов

Переименование столбцов: `.rename()`

In [16]:
# Переименовать несколько столбцов
df_renamed = df.rename(columns={'Name': 'FullName', 'Age': 'Years', 'Fare': 'Price'})
print(df_renamed.columns)

# Переименовать индекс строки
df_renamed_index = df.rename(index={0: 'Row_Zero'})
print(df_renamed_index.index[:5])

Index(['PassengerId', 'Survived', 'Pclass', 'FullName', 'Sex', 'Years',
       'SibSp', 'Parch', 'Ticket', 'Price', 'Cabin', 'Embarked',
       'PassengerId_str', 'Age_int'],
      dtype='str')
Index(['Row_Zero', 1, 2, 3, 4], dtype='object')


Изменение всех названий столбцов

In [17]:
# Перевести в верхний регистр
df_upper = df.copy()
df_upper.columns = df_upper.columns.str.upper()
print(df_upper.columns)

Index(['PASSENGERID', 'SURVIVED', 'PCLASS', 'NAME', 'SEX', 'AGE', 'SIBSP',
       'PARCH', 'TICKET', 'FARE', 'CABIN', 'EMBARKED', 'PASSENGERID_STR',
       'AGE_INT'],
      dtype='str')


### 3.2 Объединение таблиц

Вертикальное объединение: `pd.concat()` — склеивает таблицы по строкам (одну под другой).

In [18]:
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]})

# Объединение по вертикали
result = pd.concat([df1, df2], ignore_index=True)
print(result)

   A  B
0  1  3
1  2  4
2  5  7
3  6  8


Горизонтальное объединение

In [19]:
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'C': [5, 6], 'D': [7, 8]})

result = pd.concat([df1, df2], axis=1)
print(result)

   A  B  C  D
0  1  3  5  7
1  2  4  6  8


Слияние таблиц: `pd.merge()` — аналог SQL JOIN, объединяет таблицы по общему ключу.

In [20]:
# Две таблицы с общим ключом
employees = pd.DataFrame({
    'ID': [1, 2, 3, 4],
    'Name': ['Alice', 'Bob', 'Charlie', 'David']
})

salaries = pd.DataFrame({
    'ID': [1, 2, 3, 5],
    'Salary': [50000, 60000, 70000, 80000]
})

# Inner join (пересечение)
result = pd.merge(employees, salaries, on='ID', how='inner')
print(result)

   ID     Name  Salary
0   1    Alice   50000
1   2      Bob   60000
2   3  Charlie   70000


Типы слияния (`how`): `inner` — только совпадения, `outer` — все записи из обеих таблиц, `left` — все из левой, совпадения из правой, `right` — все из правой, совпадения из левой.

In [21]:
# Left join
result = pd.merge(employees, salaries, on='ID', how='left')
print(result)

   ID     Name   Salary
0   1    Alice  50000.0
1   2      Bob  60000.0
2   3  Charlie  70000.0
3   4    David      NaN


Слияние по индексу: `.join()`

In [22]:
df1 = pd.DataFrame({'A': [1, 2]}, index=['a', 'b'])
df2 = pd.DataFrame({'B': [3, 4]}, index=['a', 'b'])

result = df1.join(df2)
print(result)

   A  B
a  1  3
b  2  4
